# Guia 4 - Aprendizaje no supervisado

**Objetivo:** preparar un dataset y aplicar clustering para descubrir segmentos de clientes.

En esta guia no se usa variable objetivo `y`. La variable `Abandono` no se utiliza para entrenar los clusters.

In [ ]:
# ============================================================
# 1. Importar librerias
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

print('Librerias cargadas correctamente')

In [ ]:
# ============================================================
# 2. Cargar dataset
# ============================================================
from pathlib import Path

rutas_posibles = [
    Path('G4_base_clientes.csv'),
    Path('../03_dataset/G4_base_clientes.csv'),
    Path('/mnt/data/G4_base_clientes.csv')
]

for ruta in rutas_posibles:
    if ruta.exists():
        df = pd.read_csv(ruta)
        print('Dataset cargado desde:', ruta)
        break
else:
    raise FileNotFoundError('No se encontro G4_base_clientes.csv. Revise la ubicacion del dataset.')

print('Dimensiones:', df.shape)
df.head()

## 3. Diagnostico inicial
Revise tipos de datos, valores nulos, duplicados y valores unicos de variables categoricas.

In [ ]:
# Complete y ejecute el diagnostico
print(df.info())
print("\nNulos por columna:")
print(df.isnull().sum())
print("\nDuplicados:", df.duplicated().sum())
print("\nValores de Segmento:", df['Segmento'].unique())
print("Valores de Satisfaccion:", df['Satisfaccion'].unique())

## 4. Seleccion de variables
Seleccione variables que representen comportamiento o caracteristicas utiles para segmentar. No incluya `ID_Cliente` ni use `Abandono` como objetivo.

In [ ]:
variables_cluster = [
    'Edad', 'IngresoMensual', 'CantidadCompras', 'ComprasUltimos12M',
    'AntiguedadMeses', 'QuejasUltimos6M', 'DiasDesdeUltimaCompra',
    'VisitasWebUltimoMes', 'TiempoPromedioSesionMin', 'CuponesUsados',
    'Ciudad', 'CanalPreferido', 'ZonaResidencia', 'Segmento', 'Satisfaccion'
]

X_cluster = df[variables_cluster].copy()
X_cluster.head()

## 5. Preprocesamiento
K-Means usa distancias. Por eso se escalan numericas y se codifican categoricas.

In [ ]:
columnas_numericas = [
    'Edad', 'IngresoMensual', 'CantidadCompras', 'ComprasUltimos12M',
    'AntiguedadMeses', 'QuejasUltimos6M', 'DiasDesdeUltimaCompra',
    'VisitasWebUltimoMes', 'TiempoPromedioSesionMin', 'CuponesUsados'
]

columnas_nominales = ['Ciudad', 'CanalPreferido', 'ZonaResidencia']
columnas_ordinales = ['Segmento', 'Satisfaccion']

# Compatibilidad con versiones de scikit-learn
try:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocesador_cluster = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), columnas_numericas),
        ('nom', onehot, columnas_nominales),
        ('ord', OrdinalEncoder(
            categories=[['Basico', 'Medio', 'Premium'], ['Baja', 'Media', 'Alta']],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ), columnas_ordinales)
    ]
)

X_preparado = preprocesador_cluster.fit_transform(X_cluster)
print('Matriz preparada:', X_preparado.shape)

## 6. Metodo del codo
Pruebe diferentes valores de k y observe la inercia.

In [ ]:
inercias = []
rango_k = range(2, 9)

for k in rango_k:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    modelo.fit(X_preparado)
    inercias.append(modelo.inertia_)

plt.figure(figsize=(8,5))
plt.plot(list(rango_k), inercias, marker='o')
plt.title('Metodo del codo')
plt.xlabel('Numero de clusters k')
plt.ylabel('Inercia')
plt.grid(True)
plt.show()

pd.DataFrame({'k': list(rango_k), 'inercia': inercias})

## 7. Silhouette score
Calcule silhouette para varios valores de k.

In [ ]:
silhouettes = []
for k in rango_k:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = modelo.fit_predict(X_preparado)
    sil = silhouette_score(X_preparado, etiquetas)
    silhouettes.append(sil)

plt.figure(figsize=(8,5))
plt.plot(list(rango_k), silhouettes, marker='o')
plt.title('Silhouette score por numero de clusters')
plt.xlabel('Numero de clusters k')
plt.ylabel('Silhouette score')
plt.grid(True)
plt.show()

pd.DataFrame({'k': list(rango_k), 'silhouette': silhouettes})

## 8. Entrenar modelo final de clustering
Seleccione un valor de `k_optimo` con base en el codo, silhouette y sentido de negocio.

In [ ]:
k_optimo = 3  # Ajuste este valor segun su analisis

modelo_kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
clusters = modelo_kmeans.fit_predict(X_preparado)

df_segmentado = df.copy()
df_segmentado['Cluster'] = clusters

print(df_segmentado['Cluster'].value_counts().sort_index())
df_segmentado.head()

## 9. Perfilamiento de clusters
Calcule promedios por cluster e interprete los segmentos.

In [ ]:
perfil_numerico = df_segmentado.groupby('Cluster')[columnas_numericas].mean().round(2)
perfil_numerico

In [ ]:
# Visualizacion 2D con PCA
pca = PCA(n_components=2, random_state=42)
componentes = pca.fit_transform(X_preparado)

plt.figure(figsize=(8,6))
plt.scatter(componentes[:,0], componentes[:,1], c=clusters, alpha=0.75)
plt.title('Visualizacion de clusters con PCA')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.grid(True)
plt.show()

## 10. Exportar dataset segmentado
Al final exporte el dataset con la columna Cluster.

In [ ]:
df_segmentado.to_csv('G4_dataset_segmentado.csv', index=False, encoding='utf-8-sig')
print('Dataset segmentado exportado como G4_dataset_segmentado.csv')

## 11. Preguntas de analisis
1. Que valor de k selecciono y por que?
2. Que cluster representa clientes mas activos?
3. Que cluster puede requerir atencion comercial?
4. Que variables ayudan a diferenciar mejor los grupos?
5. Que limitaciones tiene el analisis?